[Reference](https://pub.towardsai.net/all-you-need-to-know-about-retrieval-augmented-generation-rag-in-2025-04c386284c18)

# Using LangChain

In [1]:
!pip install -qU langchain langchain-community langchain-text-splitters
!pip install -qU langchain-huggingface langchain-google-genai langchain-openai
!pip install -qU faiss-cpu pypdf fpdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.7/84.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.6/426.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [2]:
# Libraries for PDF creation
from fpdf import FPDF
import textwrap

# Libraries for RAG
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS

In [3]:
# Load documents
print("Loading documents...")
loader = PyPDFDirectoryLoader("./company_docs/")
documents = loader.load()
print(f"Loaded {len(documents)} documents (1 doc for each page)")

In [4]:
# Chunk documents
print("Chunking documents...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

In [5]:
# Create embeddings and FAISS vector store
print("Creating embeddings and FAISS vector database...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
# Save FAISS index
vectorstore.save_local("faiss_index")

# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}     # retrieve top 4 similar chunks
)

In [6]:
# Set your Google API key
os.environ["GOOGLE_API_KEY"] = "your_api_key"

# Create LLM and QA chain
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-lite",
    temperature=0
)

In [7]:
# Create RAG pipeline

def ask_with_sources(question):

    # Retrieve docs first
    docs = retriever.invoke(question)

    # Format context
    context = "\n\n".join(
    f"Source: {doc.metadata.get('source', 'Unknown')} (Page {doc.metadata.get('page', 'N/A')})\nContent: {doc.page_content}"
    for doc in docs)

    # Generate answer
    prompt_text = f"""
    Answer the question based only on the following retrieved context, and include the source used at the end as reference:

    {context}

    Question: {question}
    """
    response = llm.invoke(prompt_text)

    # Print result with sources
    print(f"\nQuestion: {question}")
    print(f"\nAnswer: {response.content}")
    print(f"\nSources Retrieved:")
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get('source', 'Unknown')
        page = doc.metadata.get('page', 'N/A')
        print(f"  {i}. {source}, Page {page}")

In [8]:
question = "what are the Ambient noise levels required"
ask_with_sources(question)

# Using LlamaIndex

In [9]:
!pip install -qU llama-index-llms-google-genai llama-index llama-index-embeddings-huggingface
!pip install -qU nest-asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 10.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [10]:
# libraries for rag
import os
import re
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.query_engine import CitationQueryEngine

import nest_asyncio # necessary for notebooks
nest_asyncio.apply()

In [11]:
# Define the Global Settings
Settings.llm = GoogleGenAI(model="models/gemini-2.0-flash-lite")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Load documents
documents = SimpleDirectoryReader('./company_docs').load_data()

# Create index
index = VectorStoreIndex.from_documents(documents)

# Create query engine
query_engine = CitationQueryEngine.from_args(
    index,
    similarity_top_k=4,
    citation_chunk_size=500,
)

In [12]:
def run_rag(query):
  response = query_engine.query(query)

  print("Answer:", response.response)
  print("="*50)

  # Extract citation numbers like [1], [4], etc.
  citations = re.findall(r'\[(\d+)\]', response.response)
  cited_indices = {int(cid) for cid in citations}  # Use set for fast lookup

  # Display only cited nodes
  for i, node in enumerate(response.source_nodes, start=1):
      if i in cited_indices:
          print(f"[{i}] Metadata (CITED):")
          print("  File:", node.metadata.get('file_name', 'Unknown'))
          print("  Page:", node.metadata.get('page_label', 'N/A'))
          print("  Score:", node.score)
          print("-" * 40)

# Test
response = run_rag("what are the Ambient noise levels required")

# Creating a RAG engine with reranking:

In [13]:
# Define Global Settings
Settings.llm = GoogleGenAI(model="models/gemini-2.0-flash-lite")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Load documents
documents = SimpleDirectoryReader('./company_docs').load_data()

# Create index
index = VectorStoreIndex.from_documents(documents)

# Set up reranker (post-processor)
rerank = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n=3  # number of final nodes to keep after reranking
)

# Create query engine with reranker
query_engine = CitationQueryEngine.from_args(
    index,
    similarity_top_k=10,      # retrieve more candidates for reranking
    citation_chunk_size=128,
    node_postprocessors=[rerank],  # apply reranking after retrieval
)

In [14]:
# Test
response = run_rag("what are the Ambient noise levels required")

# GraphRAG: Knowledge Graph Integration

In [15]:
!pip install -qU llama-index llama-index-llms-gemini llama-index-embeddings-huggingface llama-index-graph-stores-neo4j
!pip install -qU llama-index-extractors-entity sentence-transformers nest_asyncio pyvis yfiles_jupyter_graphs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.2/313.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 48.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00


In [16]:
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.graph_stores import SimplePropertyGraphStore
from llama_index.core.indices.property_graph import PropertyGraphIndex

import nest_asyncio
nest_asyncio.apply()

In [17]:
# Simple demo documents
texts = [
    "Apple Inc. is headquartered in Cupertino, California. Tim Cook is the CEO of Apple.",
    "Microsoft was founded by Bill Gates and Paul Allen. Microsoft is based in Redmond, Washington.",
    "Google is a subsidiary of Alphabet Inc. Sundar Pichai is the CEO of Google.",
    "Apple and Microsoft are competitors in the tech industry."
]

documents = [Document(text=t) for t in texts]

In [19]:
# Create an in-memory graph store
graph_store = SimplePropertyGraphStore()

# Set up LLM and embedding model settings
Settings.llm = GoogleGenAI(model="models/gemini-2.0-flash-lite")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Build index — LlamaIndex auto-extracts entities/relations using LLM
index = PropertyGraphIndex.from_documents(
    documents,
    graph_store=graph_store,
    show_progress=True,
    use_async=False,
    llm=Settings.llm,
)

# Create query engine that uses both vector + graph context
query_engine = index.as_query_engine(
    include_text=True,
    response_mode="tree_summarize",
    similarity_top_k=2,
)

In [20]:
# Ask a question that benefits from graph reasoning
question = "Who is the CEO of Apple, and where is it headquartered?"
response = query_engine.query(question)

print("Answer:\n", response.response)

# Show graph context used
print("\nGraph context used:")
for node in response.source_nodes:
    print("-", node.text)